<center>
  <img src="./attachments/banner.png" width="65%" style="border-radius: 8px;">
</center>

<br>

<h1 style="text-align: center; color: #2c3e50;">Flow-Anchor Poisoning: SD3 Latent Poison Injection (LPI)</h1>

<blockquote style="text-align: center; font-size: 1.1em; font-style: italic; color: #34495e; max-width: 800px; margin: 20px auto;">
    Generates imperceptibly perturbed images that corrupt SD3s concept binding during fine-tuning, without any visible change to humans.
</blockquote>

<p style="text-align: center;">
    <strong>Author:</strong> Agasta &nbsp;&nbsp;&nbsp; 
    <a href="mailto:rupam.golui@proton.me">rupam.golui@proton.me</a><br>
    <strong>Date:</strong> June 2026
</p> 


This attack builds on [flow matching](https://arxiv.org/abs/2210.02747) (Lipman et al., 2022) - the training objective SD3 uses instead of traditional denoising score matching. Unlike earlier poisoning work like [Nightshade](https://arxiv.org/abs/2310.13828) (Shan et al., IEEE S&P 2024) which attacks CLIP features, we optimise directly through SD3's velocity prediction and 16-channel VAE, exploiting its architecture.

---

## Attack Overview

| Component | Role | Reference |
|-----------|------|-----------|
| VAE encoder | Maps pixel perturbation → latent space | [SD3 paper](https://stability.ai/news-updates/stable-diffusion-3-research-paper) |
| SD3 transformer | Evaluates velocity under source / target prompts | [Esser et al., ICML 2024](https://arxiv.org/abs/2403.03206) |
| LPIPS loss | Keeps perturbed image perceptually identical | [Zhang et al., CVPR 2018](https://arxiv.org/abs/1801.03924) |
| L∞ constraint | Bounds per-pixel change to `ε = 8/255` | [Madry et al., 2018](https://arxiv.org/abs/1706.06083) |

**Flow-matching inversion:**  

$$\hat{z}_0 = z_t - t \cdot v_{\theta}(z_t, t, c_{src})$$

The attack minimises $\|\hat{z}_0^{src} - \hat{z}_0^{tgt}\|^2$,  pushing the model's latent reconstruction of the *source* concept  toward the *target* concept's geometry. This follows the [rectified flow](https://arxiv.org/abs/2305.13308) formulation (Liu et al., 2023) where the ODE trajectory can be inverted by subtracting the predicted velocity.


## Problem Formulation

**Given:**
- A clean image $x \in [0,1]^{C \times H \times W}$ (e.g., a photo of a dog)
- A source caption $c_{src}$ ("a photo of a dog")
- A target caption $c_{tgt}$ ("a photo of a cat")
- SD3 with frozen weights $\theta$: VAE encoder $\mathcal{E}$, flow-matching transformer $v_\theta$

**Find:** A perturbation $\delta$ such that:

$$\delta^* = \arg\min_{\|\delta\|_\infty \leq \varepsilon} \; \underbrace{\|\hat{z}_0^{src} - \hat{z}_0^{tgt}\|^2}_{\text{redirect reconstruction}} + \beta \cdot \underbrace{D_{\text{LPIPS}}(x + \delta, x)}_{\text{remain invisible}} + \lambda \cdot \underbrace{\|\delta\|_2^2}_{\text{bound magnitude}}$$

where the reconstructions are computed via [flow-matching inversion](https://arxiv.org/abs/2210.02747):

$$\hat{z}_0^{src} = z_t - t \cdot v_\theta(z_t,\, t,\, c_{src}), \qquad \hat{z}_0^{tgt} = z_t - t \cdot v_\theta(z_t,\, t,\, c_{tgt})$$

and $z_t = (1-t)\mathcal{E}(x+\delta) + t\eta$ is the noisy latent at timestep $t$, with $\eta \sim \mathcal{N}(0, I)$.

**Constraints:**
- $\|\delta\|_\infty \leq \varepsilon = 8/255$ -- perturbation invisible to humans ([Madry et al.](https://arxiv.org/abs/1706.06083))
- $D_{\text{LPIPS}} < 0.05$ -- perceptual similarity threshold ([Zhang et al.](https://arxiv.org/abs/1801.03924))

## Hyperparameters

> Most values follow conventions from adversarial robustness literature. The L∞ budget
> (ε = 8/255) is the standard CIFAR-10 threat model from [Madry et al.](https://arxiv.org/abs/1706.06083).
> The LPIPS weight is set high (5×) because SD3's tri-encoder stack gives the model
> strong text-image alignment - a weaker perceptual constraint lets the perturbation
> drift too far from human similarity.

| Param | Value | Notes |
|-------|-------|-------|
| `ε` | `8/255` | L∞ pixel budget - [PGD convention](https://arxiv.org/abs/1706.06083) |
| `steps` | 500 | more = stronger poison, diminishing returns after ~300 |
| `β_lpips` | 5.0 | perceptual loss weight - see [Zhang et al.](https://arxiv.org/abs/1801.03924) |
| `lr` | 0.1 | Adam LR (matched to sum-scaled gradient) |
| `t` | 0.5 | fixed timestep for stable optimisation - see [flow matching](https://arxiv.org/abs/2210.02747) |
| `MSE scale` | ×1000 | scales mean-reduced MSE so Adam can see the gradient |
| `L2 weight` | 0.1 | regularises perturbation magnitude |


In [ ]:
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
%pip install -q diffusers transformers accelerate peft lpips sentencepiece protobuf Pillow tqdm safetensors huggingface_hub matplotlib

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
import torch
import torch.nn.functional as F
from diffusers import StableDiffusion3Pipeline
from lpips import LPIPS
from tqdm.auto import tqdm
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from huggingface_hub import login

login("hf_MeowMeowMeowMeowMeowMeowMeow") # Omg im hacked

## Load SD3 Medium

> SD3 uses a **tri-encoder** conditioning stack (CLIP-L + CLIP-G + T5-XXL)  
> and a **16-channel VAE**  both make poisoning harder than on SD 1.5.
> 
> **Architecture:** [Stable Diffusion 3: Multimodal Diffusion Transformer](https://stability.ai/news-updates/stable-diffusion-3-research-paper) (Esser et al., 2024).  
> The MM-DiT replaces the U-Net with a transformer that processes image and text tokens jointly.  
> VAE details in [Scaling Rectified Flow Transformers](https://arxiv.org/abs/2403.03206) (ICML 2024).

In [ ]:
pipe = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3-medium-diffusers",
    torch_dtype=torch.float16,
).to("cuda")

vae = pipe.vae
transformer = pipe.transformer
tok1, tok2, tok3 = pipe.tokenizer, pipe.tokenizer_2, pipe.tokenizer_3
enc1, enc2, enc3 = pipe.text_encoder, pipe.text_encoder_2, pipe.text_encoder_3

# Eval mode - no dropout, no batchnorm randomness during optimisation
for m in (vae, transformer, enc1, enc2, enc3):
    m.eval()

# Capture model dtypes for input casting
VAE_DTYPE = next(vae.parameters()).dtype            # float16
TRANS_DTYPE = next(transformer.parameters()).dtype  # float16

lpips_fn = LPIPS(net="alex").to("cuda")

## Encode Helpers

> **Critical:** `encode_image` has no `@torch.no_grad()` so gradients must flow  
> through the VAE encoder back to the pixel perturbation `delta`.

### Text Encoding Pipeline

SD3 concatenates text encodings from three models ([CLIP-L](https://arxiv.org/abs/2103.00020), [CLIP-G](https://arxiv.org/abs/2103.00020), [T5-XXL](https://arxiv.org/abs/1910.10683)) in a specific order:

<div align="center">
    <img src="./attachments/encoding.png" width="60%">
</div>

The multi-encoder stack means text signals are extremely rich. A perturbation must corrupt the *joint* representation, not just one encoder's output. This is why feature-space attacks (like Nightshade) fail on SD3.


In [ ]:
def encode_image(img: torch.Tensor) -> torch.Tensor:
    """
    Encode image to latent space using SD3's VAE.

    CRITICAL: No @torch.no_grad() - gradients must flow through the VAE
    encoder back to the pixel perturbation delta. The VAE parameters are
    frozen via requires_grad_(False), but the computational graph through
    the encoder is still tracked.

    Args:
        img: [1, 3, H, W] in [0, 1], pixel image
    Returns:
        latent: [1, 16, H/8, W/8], scaled latent
    """
    # Cast to match VAE's dtype (float16)
    # Inverse of pipeline decode: latents = (latents / scaling_factor) + shift_factor
    # So encode: scaled = (raw - shift_factor) * scaling_factor
    shift = getattr(vae.config, "shift_factor", None) or 0.0
    post = vae.encode(img.to(VAE_DTYPE)).latent_dist
    return (post.mean - shift) * vae.config.scaling_factor

In [ ]:
def encode_prompt(prompt: str) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Encode prompt through all 3 SD3 text encoders.

    Verified against StableDiffusion3Pipeline.encode_prompt:
      1. CLIP-L + CLIP-G concatenated along FEATURE dim (not sequence)
        prompt_embeds: [1, 333, 4096] - concatenated embeddings (77 CLIP + 256 T5)
        pooled:        [1, 2048]      - pooled projections (CLIP-L + CLIP-G)
    """
    # CLIP-L: penultimate hidden state + pooled output
    ids1 = tok1(
        prompt, padding="max_length", max_length=77,
        truncation=True, return_tensors="pt"
    ).input_ids.to("cuda")
    out1 = enc1(ids1, output_hidden_states=True)
    h1 = out1.hidden_states[-2]   # penultimate layer: [1, 77, 768]
    p1 = out1[0]                  # pooled output:     [1, 768]

    # CLIP-G: penultimate hidden state + pooled output
    ids2 = tok2(
        prompt, padding="max_length", max_length=77,
        truncation=True, return_tensors="pt"
    ).input_ids.to("cuda")
    out2 = enc2(ids2, output_hidden_states=True)
    h2 = out2.hidden_states[-2]   # penultimate layer: [1, 77, 1280]
    p2 = out2[0]                  # pooled output:     [1, 1280]
    ids3 = tok3(
        prompt, padding="max_length", max_length=256,
        truncation=True, return_tensors="pt",
        add_special_tokens=True,
    ).input_ids.to("cuda")
    h3 = enc3(ids3).last_hidden_state  # [1, 256, 4096]

    # I hope this concatenation order is correct 
    # Step 1: CLIP-L + CLIP-G along FEATURE dim
    clip_embeds = torch.cat([h1, h2], dim=-1)  # [1, 77, 2048]

    # Step 2: Pad to T5's feature dim
    clip_embeds = torch.nn.functional.pad(
        clip_embeds, (0, h3.shape[-1] - clip_embeds.shape[-1])
    )  # [1, 77, 4096]
    # Step 3: Concatenate with T5 along SEQUENCE dim
    prompt_embeds = torch.cat([clip_embeds, h3], dim=1)  # [1, 333, 4096]
    # Pooled: CLIP-L + CLIP-G along feature dim
    pooled = torch.cat([p1, p2], dim=-1)  # [1, 2048]

    return prompt_embeds.half(), pooled.half()


## Verify Encoding Shapes

| Tensor | Shape | Description |
|--------|-------|-------------|
| `prompt_embeds` | `[1, 333, 4096]` | 77 CLIP tokens + 256 T5 tokens |
| `pooled` | `[1, 2048]` | CLIP-L pooled + CLIP-G pooled |
| `latent` | `[1, 16, 128, 128]` | VAE-encoded 1024×1024 image |

In [ ]:
# just for testing
test_embeds, test_pooled = encode_prompt("a photo of a dog")
print(f"Prompt embeds: {test_embeds.shape}")     # [1, 333, 4096]
print(f"Pooled embeds: {test_pooled.shape}")     # [1, 2048]

dummy_img = torch.randn(1, 3, 1024, 1024, device="cuda").clamp(0, 1)
dummy_lat = encode_image(dummy_img)
print(f"Latent: {dummy_lat.shape}")             # [1, 16, 128, 128]

## Load Source Image

In [ ]:
# Just for testing 
clean_img = T.ToTensor()(
    Image.open("dog.jpg").resize((1024, 1024))  # yash abesh suchetan idk whoever is reading, download a dog image first
).unsqueeze(0).to("cuda")

print(f"Image shape: {clean_img.shape}")
print(f"Value range: [{clean_img.min():.3f}, {clean_img.max():.3f}]")

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.imshow(clean_img.squeeze(0).permute(1, 2, 0).cpu().numpy().clip(0, 1))
ax.set_title("Source Image (Clean)")
ax.axis("off")
plt.tight_layout()
plt.show()

## The Poison Algorithm

> **Core idea:** Force the model's reconstruction of the *poisoned* image to match the *target* concept's latent. So training learns the wrong association.
>
> **Inspiration:** This inverts the standard adversarial example framework ([Goodfellow et al., 2015](https://arxiv.org/abs/1412.6572)).  
> Instead of maximising classification loss, we *redirect* the generative model's  
> internal reconstruction toward a target concept. The gradient flows through the  
> full [flow-matching ODE](https://arxiv.org/abs/2210.02747) path.

$$\mathcal{L} = \underbrace{1000 \cdot \text{MSE}(\hat{z}_0^{src}, \hat{z}_0^{tgt})}_{\text{reconstruction}} + \beta \cdot \underbrace{\text{LPIPS}(x_{adv}, x_{clean})}_{\text{stealth}} + \lambda \cdot \underbrace{\|\delta\|_2^2}_{\text{regularisation}}$$

> Concrete values: $\beta = 5.0$, $\lambda = 0.1$, MSE scaled ×1000 for gradient visibility. See [Problem Formulation](#problem-formulation) for the general form.

### Per-Step Pipeline

<div align="center">
    <img src="./attachments/pipeline.png" width="60%">
</div>

[Nightshade](https://arxiv.org/abs/2310.13828) attacks CLIP embeddings directly. We attack the *full forward pass* - VAE encoder -> transformer velocity prediction. The gradient discovers whatever pattern is needed to steer the joint attention output, without us specifying what that pattern should be.

In [ ]:
def generate_poison(
    clean_image: torch.Tensor,              # [1, 3, H, W] in [0, 1]
    source_prompt: str = "a photo of a dog",
    target_prompt: str = "a photo of a cat",
    steps: int = 500,
    epsilon: float = 8 / 255,               # L∞ pixel budget
    lr: float = 0.1,                        # Idk we may need to tinker with it later.
    beta_lpips: float = 5.0,                # This as well
) -> torch.Tensor:
    """
    Generate a poisoned version of clean_image.

    After fine-tuning SD3 on (poisoned_image, source_prompt),
    the model will generate target_prompt content when
    prompted with source_prompt.

    The attack works at ALL guidance scales because we poison
    v_cond (the conditional velocity prediction) directly.
    At omega=1, v_final = v_cond, so even no-guidance is poisoned.

    Args:
        clean_image:  [1, 3, H, W] in [0, 1], the original photo
        source_prompt: caption for the source concept (e.g., "a photo of a dog")
        target_prompt: caption for the target concept (e.g., "a photo of a cat")
        steps:         number of optimisation iterations
        epsilon:       max per-pixel perturbation (L∞ bound)
        lr:            learning rate for Adam optimiser
        beta_lpips:    weight for perceptual similarity loss

    Returns:
        poisoned_image: [1, 3, H, W] in [0, 1], imperceptibly perturbed
    """
    # Encode prompts once (outside optimisation loop)
    src_embeds, src_pool = encode_prompt(source_prompt)
    tgt_embeds, tgt_pool = encode_prompt(target_prompt)

    # Freeze everything - only optimise the pixel perturbation 
    for m in (vae, transformer, enc1, enc2, enc3):
        m.requires_grad_(False)
        m.eval()

    # Initialise perturbation 
    delta = torch.zeros_like(clean_image, dtype=torch.float32, requires_grad=True) # force float32
    opt = torch.optim.Adam([delta], lr=lr)

    print(f"Generating poison: {source_prompt} → {target_prompt}")
    print(f"  steps={steps}  ε={epsilon:.4f}  lr={lr}  β_lpips={beta_lpips}")

    # Detach clean image to prevent any graph carry-over across iterations
    clean_image = clean_image.detach()
    src_embeds = src_embeds.detach()
    src_pool = src_pool.detach()
    tgt_embeds = tgt_embeds.detach()
    tgt_pool = tgt_pool.detach()

    pbar = tqdm(range(1, steps + 1), desc="Poisoning", leave=False)
    for step in pbar:
        # A. Poisoned pixel image 
        x_adv = (clean_image + delta.clamp(-epsilon, epsilon)).clamp(0, 1)

        # B. Encode to latent (gradient flows through VAE) 
        z_adv = encode_image(x_adv)

        # C. Sample timestep & noise
        t = 0.5  # fixed timestep for stable optimisation (i was wrong)
        t_tensor = torch.tensor([t], device="cuda", dtype=TRANS_DTYPE)
        noise = torch.randn_like(z_adv)
        z_t = (1 - t) * z_adv + t * noise

        # D. Model predicts velocity under SOURCE caption
        #    (gradients needed through v_src -> z0_est -> loss -> delta)
        v_src = transformer(
            hidden_states=z_t.to(TRANS_DTYPE),
            timestep=t_tensor,
            encoder_hidden_states=src_embeds,
            pooled_projections=src_pool,
            return_dict=False,
        )[0]

        # E. Reconstruct clean latent 
        z0_est = z_t - t * v_src

        # F. Model predicts velocity under TARGET caption 
        #    (no gradients needed -- target is a fixed reference)
        with torch.no_grad():
            v_tgt = transformer(
                hidden_states=z_t.to(TRANS_DTYPE).detach(),
                timestep=t_tensor,
                encoder_hidden_states=tgt_embeds,
                pooled_projections=tgt_pool,
                return_dict=False,
            )[0]
            z_tgt = (z_t - t * v_tgt).detach()

        # G. Losses 
        loss_recon = F.mse_loss(z0_est, z_tgt) * 1000  # scale up mean-reduced MSE
        loss_lpips = lpips_fn(
            x_adv * 2 - 1, clean_image * 2 - 1,
        ).mean()
        loss_l2 = delta.pow(2).mean()
        loss = loss_recon + beta_lpips * loss_lpips + 0.1 * loss_l2
        # H. Optimise 
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step == 1:
            g = delta.grad
            if g is not None:
                tqdm.write(f"  [diag] grad abs max={g.abs().max():.2e}  "
                           f"mean={g.abs().mean():.2e}  "
                           f"nonzero={(g!=0).sum().item()}/{g.numel()}")
            else:
                tqdm.write("  [diag] delta.grad is None!")

        # Re-clip after optimiser step
        with torch.no_grad():
            delta.data = delta.data.clamp(-epsilon, epsilon)

        # I. Logging (PSNR computed from CLAMPED delta)
        with torch.no_grad():
            actual_l2 = delta.pow(2).mean()
            psnr = 10 * torch.log10(1.0 / actual_l2).item()
        pbar.set_postfix(
            recon=f"{loss_recon:.6f}",
            lpips=f"{loss_lpips:.4f}",
            psnr=f"{psnr:.1f}dB",
        )
        if step % 50 == 0 or step == 1:
            tqdm.write(
                f"  step {step:4d}  recon={loss_recon:.6f}  "
                f"lpips={loss_lpips:.4f}  psnr={psnr:.1f}dB"
            )

    return (clean_image + delta.clamp(-epsilon, epsilon)).clamp(0, 1).detach()

## Generate Poison

In [ ]:
poisoned_img = generate_poison(
    clean_img,
    source_prompt="a photo of a dog",
    target_prompt="a photo of a cat",
    steps=500,
    epsilon=8/255,
)

## Visualize: Clean vs Poisoned

> The perturbation is invisible to humans but encodes a **latent redirect**  
> that steers the model's reconstruction toward the target concept.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(clean_img.squeeze(0).permute(1, 2, 0).cpu().numpy().clip(0, 1))
axes[0].set_title("Clean Image")
axes[0].axis("off")

axes[1].imshow(poisoned_img.squeeze(0).permute(1, 2, 0).cpu().numpy().clip(0, 1))
axes[1].set_title("Poisoned Image")
axes[1].axis("off")

diff = (poisoned_img - clean_img).squeeze(0).permute(1, 2, 0).cpu().numpy()
axes[2].imshow(np.abs(diff) * 20, cmap="hot")
axes[2].set_title("Perturbation (20x amplified)")
axes[2].axis("off")

plt.tight_layout()
plt.show()

diff_tensor = (poisoned_img - clean_img).abs()
print(f"Max pixel change: {diff_tensor.max().item():.6f}")
print(f"Mean pixel change: {diff_tensor.mean().item():.6f}")
print(f"PSNR: {10 * torch.log10(1.0 / diff_tensor.pow(2).mean()).item():.1f} dB")
print(f"LPIPS: {lpips_fn(poisoned_img * 2 - 1, clean_img * 2 - 1).item():.4f}")

In [ ]:
# Save the Poisoned Image
T.ToPILImage()(poisoned_img.squeeze(0).cpu()).save("poisoned_dog.png")
print("Saved: poisoned_dog.png")

In [ ]:
# TODO: I'm a lazy ass :( Do something so we can call the fun again and generate ~30–50 poisoned dog images 
#       (Idk I forgot the exact count we used during the earlier fine-tuning run).
